# Prepare meteo data for submission

## 2020
Data are hourly and have been flagged.

## 2021
Data prior to 1 Setember are hourly, aterwards, there are some 10' data that need to be aggregated.

## 2023 - 2024
The 2m Lufft instrument was destroyed and not available between mid-October 2023 and April 2024. 
The Rotronic data are available for the entire period, but are not aggregated on the DWH (why not?). 
We will use the official hourly data from DWH where available and complement with our own aggregated Rotronic data where needed.

author: joerg.klausen@meteoswiss.ch

In [1]:
import os
import polars as pl
from processing.dwh import DWH
from toolbox.utils import pl_simplify_dtypes

access_token = open(file="secrets-jretrieve", mode="r").read()
mkn = DWH(access_token=access_token, locationID="KEMKN")

DWH initialized.


In [2]:
metadata = {
    'prestah0': '2m pressure (QFE), Lufft (hPa)',
    'ta2200h0': '2m temperature, Rotronic (°C)',
    'ua2200h0': '2m relative humidity, Rotronic (%)',
    'fkl010h0': '10m horizontal wind speed (m/s)',
    'dkl010h0': '10m horizontal wind direction (deg)',
    'rre150h0': '2m precipitation, Lufft (mm/h)',
    'gre000h0': '2m global radiation, Lufft (W/m2)',
    # 'tre200h0': '2m temperature, Lufft (up until 19 Oct 2023 09 UTC)',
    # 'ure200h0': '2m relative humidity, Lufft (%) (up until 19 Oct 2023 09 UTC)',
}

In [ ]:
year_min = 2020
year_max = 2022
# get hourly values from MeteoSwiss DWH
parameter_short_names = "prestah0,tre200h0,ure200h0,fkl010h0,dkl010h0,rre150h0,gre000h0,gor000z0"
df1_1h = mkn.jretrieve(start=f"{year_min}0101000000", end=f"{year_max + 1}0101000000", parameter_short_names=parameter_short_names)
df1_1h.drop_in_place('station')
df1_1h.drop_in_place('termin')

# get Rotronic data and temperature, RH data from 10m sensor from MeteoSwiss DWH and aggregate
parameter_short_names = "ta2200s0,ua2200s0,ta1200s0,ua1200s0"
df2 = mkn.jretrieve(start=f"{year_min}0101000000", end=f"{year_max+1}0101000000", parameter_short_names=parameter_short_names)

df2_1h = df2.sort(by='dtm').group_by_dynamic("dtm", every='1h', closed='right', label='right').agg(pl.all().exclude(['dtm', 'termin', 'station']).mean())
df2_1h = df2_1h.rename({'ta2200s0': 'ta2200h0', 'ua2200s0': 'ua2200h0', 'ta1200s0': 'ta1200h0', 'ua1200s0': 'ua1200h0'})

In [ ]:
df1_1h = pl_simplify_dtypes(df=df1_1h)
df2_1h = pl_simplify_dtypes(df=df2_1h)

In [ ]:
display(df1_1h.schema)
display(df2_1h.schema)

In [ ]:
# combine dataframes, compute biases
df_1h = pl.concat([df1_1h, df2_1h], how='align')
display(df_1h)
df_1h = df_1h.with_columns((pl.col('tre200h0')-pl.col('ta2200h0')).alias('tre-ta2'))
df_1h = df_1h.with_columns((pl.col('ta1200h0')-pl.col('ta2200h0')).alias('ta1-ta2'))
df_1h = df_1h.with_columns((pl.col('ure200h0').cast(pl.Float32)-pl.col('ua2200h0').cast(pl.Float32)).alias('ure-ua2'))
df_1h = df_1h.with_columns((pl.col('ua1200h0').cast(pl.Float32)-pl.col('ua2200h0').cast(pl.Float32)).alias('ua1-ua2'))
display(df_1h.describe())

In [ ]:
# remove invalid radiation data for period where radiation sensor was not present
start = pl.lit("2024-01-01 00:00").str.to_datetime(format="%Y-%m-%d %H:%M", time_zone="UTC")
end = pl.lit("2024-07-24 23:59").str.to_datetime(format="%Y-%m-%d %H:%M", time_zone="UTC")

# Masking values in 'value1' for the given time range
df_1h = df_1h.with_columns(
    pl.when((pl.col("dtm") >= start) & (pl.col("dtm") <= end))
    .then(None)
    .otherwise(pl.col("gre000h0"))
    .alias("gre000h0")  # Replace column
)


In [ ]:
# plot data
import matplotlib.pyplot as plt
%matplotlib widget

fig, axs = plt.subplots(figsize=(10, 20), nrows=9, sharex=True)
axs[0].scatter(x=df_1h['dtm'], y=df_1h['prestah0'], s=5, label='prestah0')
axs[0].legend()
axs[1].scatter(x=df_1h['dtm'], y=df_1h['ta2200h0'], s=5, label='ta2200h0')
axs[1].scatter(x=df_1h['dtm'], y=df_1h['tre200h0'], s=5, c='r', label='tre200h0')
axs[1].scatter(x=df_1h['dtm'], y=df_1h['ta1200h0'], s=5, c='g', label='ta1200h0')
axs[1].legend()
axs[2].scatter(x=df_1h['dtm'], y=df_1h['tre-ta2'], s=5, c='c', label='tre200h0 - ta2200h0')
axs[2].scatter(x=df_1h['dtm'], y=df_1h['ta1-ta2'], s=5, c='m', label='ta1200h0 - ta2200h0')
axs[2].axhline()
axs[2].legend()
axs[3].scatter(x=df_1h['dtm'], y=df_1h['ua2200h0'], s=5, label='ua2200h0')
axs[3].scatter(x=df_1h['dtm'], y=df_1h['ure200h0'], s=5, c='r', label='ure200h0')
axs[3].scatter(x=df_1h['dtm'], y=df_1h['ua1200h0'], s=5, c='g', label='ua1200h0')
axs[3].legend()
axs[4].scatter(x=df_1h['dtm'], y=df_1h['ure-ua2'], s=5, c='c', label='ure200h0 - ua2200h0')
axs[4].scatter(x=df_1h['dtm'], y=df_1h['ua1-ua2'], s=5, c='m', label='ua1200h0 - ua2200h0')
axs[4].axhline()
axs[4].legend()
axs[5].scatter(x=df_1h['dtm'], y=df_1h['fkl010h0'], s=5, label='fkl010h0')
axs[5].legend()
axs[6].scatter(x=df_1h['dtm'], y=df_1h['dkl010h0'], s=5, label='dkl010h0')
axs[6].legend()
axs[7].scatter(x=df_1h['dtm'], y=df_1h['gre000h0'], s=5, label='gre000h0')
axs[7].legend()
axs[8].scatter(x=df_1h['dtm'], y=df_1h['rre150h0'], s=5, label='rre150h0')
axs[8].legend()

In [ ]:
# flag 

In [ ]:
# drop original columns, retain combined columns, save to file
df_1h = df_1h.select([pl.col('dtm'), pl.col('prestah0'), pl.col('ta2200h0'), pl.col('ua2200h0'),
                      pl.col('fkl010h0'), pl.col('dkl010h0'), pl.col('gre000h0'), pl.col('rre150h0'),
                      ])
display(df_1h.describe())

file = "mkn_meteo_1h"
if year_min == year_max:
    path = f"data/level2/mkn/{year_min}"    
else:
    path = f"data/level2/mkn/"
    file = f"{file}_{year_min}-{year_max}"
os.makedirs(path, exist_ok=True)

df_1h.write_parquet(os.path.join(path, f"{file}.parquet"))
df_1h.write_csv(os.path.join(path, f"{file}.csv"))

In [ ]:
# add metadata file
import json
with open(file=os.path.join(path, f"{file}.json"), mode='w') as fh:
    fh.write(json.dumps(metadata))

In [ ]:
fig, axs = plt.subplots(figsize=(10, 18), nrows=7, sharex=True, constrained_layout=True)
axs[0].scatter(x=df_1h['dtm'], y=df_1h['prestah0'], s=5, label='prestah0')
axs[0].legend()
axs[1].scatter(x=df_1h['dtm'], y=df_1h['ta2200h0'], s=5, label='ta2200h0')
axs[1].legend()
axs[2].scatter(x=df_1h['dtm'], y=df_1h['ua2200h0'], s=5, label='ua2200h0')
axs[2].legend()
axs[3].scatter(x=df_1h['dtm'], y=df_1h['fkl010h0'], s=5, label='fkl010h0')
axs[3].legend()
axs[4].scatter(x=df_1h['dtm'], y=df_1h['dkl010h0'], s=5, label='dkl010h0')
axs[4].legend()
axs[5].scatter(x=df_1h['dtm'], y=df_1h['gre000h0'], s=5, label='gre000h0')
axs[5].legend()
axs[6].scatter(x=df_1h['dtm'], y=df_1h['rre150h0'], s=5, label='rre150h0')
axs[6].legend()

In [ ]:
# load data from .parquet file. These are the data compiled from the original bulletins
# file = "data/level2/2023/vrxa00.parquet"
# df2 = pl.read_parquet(file)
# vrxa00_stats = df2.describe()
# vrxa00_stats